# Gradient term: calibration from the astromon database

From an astromon database to the gradient term's numbers. The code is in
`astrometry_corr_gradient/calibration.py`, one function per step:

1. **Matched sources:** the X-ray to catalog cross matches, rebased onto each
   detector's latest CALALIGN matrix, with the calibration's cuts.
2. **Dwells:** each observation's dwell, from kadi's maneuvers.
3. **Calibration set:** one row per source, with OOBAGRD3 and OOBAGRD6
   averaged over each dwell.
4. **Solve:** the coefficients, sigmas and reference table.

The only setting is `DATABASE`, the astromon database (only read). The
CALALIGN matrices are this repo's copy. The telemetry comes from cheta and
the safing intervals from kadi and cheta, both cached in `cache/` so a rerun
fetches only what is new.

The calibration set is saved as `gradient_term_calibration_set.ecsv`;
`python scripts/update_gradient_term_data.py` then writes the numbers into
the module.


In [1]:
import os
import subprocess
import sys
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
from astropy import units as u
from astropy.table import Table
from astropy.time import Time

os.environ.setdefault("SKA", "/Users/jean/ska")
# this checkout's package and scripts, before any installed copy
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(1, str(Path.cwd().parent / "scripts"))
from update_gradient_term_data import RELEASE_QUARTER

from astrometry_corr_gradient import calibration, gradient_term

DATABASE = "/Users/jean/git/astrometry_scratch_20260929/median_bins_telemetry/snapshots/astromon_gaussian_redo_20261122_2132.h5"
CACHE_DIR = Path.cwd().parent / "cache"
CALIBRATION_PATH = Path("gradient_term_calibration_set.ecsv")
START = "2004:001"
STOP = (
    Time.now() + 5 * u.day
).yday  # a few days past now, so the newest observations are covered
print(f"database: {DATABASE}")
print(f"cache: {CACHE_DIR}")

database: /Users/jean/git/astrometry_scratch_20260929/median_bins_telemetry/snapshots/astromon_gaussian_redo_20261122_2132.h5
cache: /Users/jean/git/astrometry_corr_gradient/cache


## 1. Matched sources


In [2]:
matches = calibration.matched_sources(DATABASE)
print(f"{len(matches):,} matched sources from {calibration.FIRST_YEAR} on")
print(
    "rebased onto:",
    {
        detector: row["start"]
        for detector, row in matches.meta["calalign_reference"].items()
    },
)

9,259 matched sources from 2004 on
rebased onto: {'ACIS-I': '2021-07-02', 'ACIS-S': '2021-07-02', 'HRC-I': '2021-07-02', 'HRC-S': '2021-07-02'}


## 2. Dwells


In [3]:
windows = calibration.dwell_windows(START, STOP)
print(f"{len(windows):,} observations with one dwell, {START} to {STOP[:8]}")

41,764 observations with one dwell, 2004:001 to 2026:332


## 3. Calibration set


In [4]:
series = {
    name: calibration.gradient_series(name, START, STOP, CACHE_DIR)
    for name in gradient_term.GRADIENT_MSIDS
}
bad_intervals = calibration.bad_intervals(
    START, STOP, CACHE_DIR / "bad_telemetry_intervals.ecsv"
)
calibration_table = calibration.calibration_set(matches, windows, series, bad_intervals)
print(
    f"{len(calibration_table):,} sources in {len(np.unique(calibration_table['obsid'])):,} observations,"
    f" {calibration_table['date'][0][:10]} to {calibration_table['date'][-1][:10]}"
)
print(
    {
        detector: int((calibration_table["detector"] == detector).sum())
        for detector in np.unique(calibration_table["detector"])
    }
)
calibration_table[:5].pprint(max_width=-1)

8,827 sources in 7,007 observations, 2004-01-03 to 2026-11-04
{np.str_('ACIS-I'): 2027, np.str_('ACIS-S'): 6119, np.str_('HRC-I'): 364, np.str_('HRC-S'): 317}
obsid         date               year        detector    target   linked_group     dz_rebased          dy_rebased           OOBAGRD3            OOBAGRD6          dz_caldb_shift       dy_caldb_shift   
                                                                                     arcsec              arcsec                K                   K                  arcsec               arcsec       
----- ------------------- ------------------ -------- ----------- ------------ ------------------ -------------------- ------------------ -------------------- ------------------- ---------------------
 4493 2004-01-03 05:59:30  2004.006146592289   ACIS-I   HIP 42540            0 0.6434212520892686 -0.05389475375655195 0.8611631285060536 0.041033241897821426 -0.8568344906146592 -0.026678328213662894
 4845 2004-01-03 07:26:38  2004.00631

## Check: against the previous calibration set

The calibration set saved last time, if there is one: which sources came or
went, and whether the common ones changed. A new database adds sources; the
same database should give the same set.


In [5]:
if CALIBRATION_PATH.exists():
    previous = Table.read(CALIBRATION_PATH, format="ascii.ecsv")
    print(f"previous: {len(previous):,} sources, made {previous.meta.get('made')}")

    def keys(table: Table) -> list[tuple]:
        return [
            (int(row["obsid"]), str(row["date"]), round(float(row["dz_rebased"]), 6))
            for row in table
        ]

    new_keys, old_keys = keys(calibration_table), keys(previous)
    print(
        f"only now: {len(set(new_keys) - set(old_keys))}, only before: {len(set(old_keys) - set(new_keys))}"
    )
    position = {key: index for index, key in enumerate(old_keys)}
    common = [
        (index, position[key]) for index, key in enumerate(new_keys) if key in position
    ]
    if common:
        here, there = (np.array(indexes) for indexes in zip(*common, strict=True))
        for name in ("dz_rebased", "dy_rebased", "OOBAGRD3", "OOBAGRD6"):
            now, before = (
                np.asarray(calibration_table[name], float)[here],
                np.asarray(previous[name], float)[there],
            )
            finite = np.isfinite(now) & np.isfinite(before)
            largest = (
                float(np.max(np.abs(now[finite] - before[finite])))
                if finite.any()
                else 0.0
            )
            same_missing = np.array_equal(np.isnan(now), np.isnan(before))
            print(
                f"{name}: {len(common):,} common sources, largest change {largest:.2e}, same missing values: {same_missing}"
            )
else:
    print("no previous calibration set")

previous: 8,289 sources, made 2026-11-23 01:56 UTC
only now: 556, only before: 18
dz_rebased: 8,271 common sources, largest change 0.00e+00, same missing values: True
dy_rebased: 8,271 common sources, largest change 0.00e+00, same missing values: True
OOBAGRD3: 8,271 common sources, largest change 0.00e+00, same missing values: True
OOBAGRD6: 8,271 common sources, largest change 0.00e+00, same missing values: True


## Saved

The calibration set, with where it came from in its header.


In [6]:
def git(*arguments: str) -> str:
    return subprocess.run(
        ["git", *arguments], capture_output=True, text=True, check=True
    ).stdout.strip()


changed = git("status", "--porcelain", "--", "..")
calibration_table.meta.update(
    {
        "description": "the gradient term's calibration set, one row per matched source (gradient_term_calibration.ipynb)",
        "made": datetime.now(UTC).strftime("%Y-%m-%d %H:%M UTC"),
        "database": str(DATABASE),
        "telemetry_last_sample": {
            name: Time(series[name][0][-1], format="cxcsec").iso[:16]
            for name in gradient_term.GRADIENT_MSIDS
        },
        "dwells": f"kadi maneuvers {START} to {STOP}",
        "git_commit": git("rev-parse", "HEAD"),
        "git_changes": changed.splitlines() if changed else "none",
    }
)
calibration_table.write(CALIBRATION_PATH, format="ascii.ecsv", overwrite=True)
reread = Table.read(CALIBRATION_PATH, format="ascii.ecsv")
for name in calibration_table.colnames:
    written, read_back = np.asarray(calibration_table[name]), np.asarray(reread[name])
    if written.dtype.kind == "f":
        assert np.array_equal(written, read_back, equal_nan=True), name
    else:
        assert list(written) == list(read_back), name
print(
    f"wrote {CALIBRATION_PATH}: {len(reread):,} sources, {len(reread.colnames)} columns; it reads back the same"
)

wrote gradient_term_calibration_set.ecsv: 8,827 sources, 12 columns; it reads back the same


## 4. Solve, and compare with the module

The coefficients and sigmas from every source with both gradients; the
reference table from one value per observation. History uses each quarter's
own medians; from the release quarter on, the previous quarter's. The quarter
now filling sets no reference yet.


In [7]:
last_complete_quarter = int(gradient_term.quarter_index(Time.now())[0]) - 1
coefficients, sigmas, references = calibration.solve(
    calibration_table,
    last_complete_quarter=last_complete_quarter,
    lagging_from=gradient_term.label_index(RELEASE_QUARTER),
)
rows = []
for axis_name in gradient_term.AXES:
    for name in gradient_term.GRADIENT_MSIDS:
        rows.append(
            {
                "axis": axis_name,
                "gradient": name,
                "c now (arcsec per K)": round(coefficients[axis_name][name], 3),
                "c in the module": gradient_term.COEFFICIENTS[axis_name][name],
                "s now (K)": round(sigmas[name], 5),
                "s in the module": gradient_term.SIGMAS[name],
            }
        )
Table(rows=rows).pprint(max_width=-1)
embedded = gradient_term.reference_table()
print(
    f"reference table: {len(references)} quarters now ({references['quarter'][0]} to {references['quarter'][-1]}),"
    f" {len(embedded)} in the module"
)
common_quarters = sorted(set(references["quarter"]) & set(embedded["quarter"]))
for name in gradient_term.GRADIENT_MSIDS:
    now = np.array(
        [
            references[name][list(references["quarter"]).index(q)]
            for q in common_quarters
        ]
    )
    module = np.array(
        [embedded[name][list(embedded["quarter"]).index(q)] for q in common_quarters]
    )
    print(
        f"{name}: largest difference over {len(common_quarters)} quarters {np.max(np.abs(now - module)):.1e} K"
    )

axis gradient c now (arcsec per K) c in the module s now (K) s in the module
---- -------- -------------------- --------------- --------- ---------------
  dz OOBAGRD3                0.475           0.462    0.1117         0.10747
  dz OOBAGRD6              -13.581         -13.284   0.00737         0.00726
  dy OOBAGRD3                1.559           1.579    0.1117         0.10747
  dy OOBAGRD6              -11.532         -11.185   0.00737         0.00726
reference table: 92 quarters now (2004Q1 to 2026Q4), 92 in the module
OOBAGRD3: largest difference over 92 quarters 1.1e-01 K
OOBAGRD6: largest difference over 92 quarters 3.1e-03 K


## Notes

- **To make new numbers:** point `DATABASE` at a newer database and rerun,
  then `python scripts/update_gradient_term_data.py`.
- **A new CALALIGN file:** `python scripts/update_calalign_table.py
  /path/to/calalign` rewrites the repo's copy of the matrices.
- **Not here yet:** averaging the gradients over each observation's GTI span
  instead of its dwell.
